# NB2 · Veriyi hazırlamak

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapılıyor

NB1'de veriye ulaştınız. Ham tablo modele doğrudan verilemez; üç iş daha gerekir.

Önce veri temizlenir: Eksik ve imkânsız değerler ele alınır. Sonra veri iki gruba
ayrılır; biriyle model öğretilir, diğeriyle sınanır. En son, modelin anlayacağı biçime
çevrilir.

Sıralama önemlidir ve üçüncü adımın en sonda olması tesadüf değildir. Sebebini üçüncü
adımda göreceksiniz.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda toplanan bloğun tamamını aşağıdaki hücreye yapıştırınız.
İlk satırdaki `#@cdss` işaretini silmeyiniz; o blok bu defterin sonunda yeniden
toplanacak ve bir sonrakine taşınacaktır.

Blok çalıştığında önceki defterlerde yazdığınız her şey yeniden kurulur. İnternetten
veri okuyan satırlar varsa bu hücre birkaç saniye sürebilir.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol · Gelen kod


In [ ]:
kit.check_defined('pd', 'np', 'kohort', 'KARAR_PENCERESI_SAAT', 'HEDEF_ESIK_GUN')


In [ ]:
kit.check_frame(kohort, name='kohort', required=['hasta_id', 'hedef'], min_rows=30)


---

## Adım 1 · Temizleme

Gerçek hastane verisi eksiksiz gelmez. Bazı ölçümler hiç yapılmamıştır, bazıları yanlış
girilmiştir, bazı kayıtlar yinelenmiştir.

Bu adımda üç iş yaptıracaksınız. Tamamen boş sütunlar atılır; içinde hiçbir bilgi
olmayan bir sütun modele yük olmaktan başka bir şey yapmaz. Yinelenen satırlar
temizlenir. Fizyolojik olarak imkânsız değerler eksik sayılır.

Son madde dikkat ister. Yaşı 200 yazılmış bir hastayı silmek yerine o değeri boş
bırakırız, çünkü hastanın diğer bilgileri hâlâ kullanılabilir. Hangi değerin imkânsız
sayılacağı klinik bir karardır ve bunu isteme siz yazarsınız.


### İstem 1

```
kohort tablosunu temizleyen bir işlem parçası yaz. Adı veriyi_temizle olsun, kendisine
verilen tabloyu temizlesin ve temizlenmiş tabloyu geri versin.

Şunları yapsın:
1. Tamamen boş olan sütunları at.
2. Birebir aynı olan yinelenen satırları at.
3. Yaş bilgisi 0'dan küçük ya da 120'den büyükse o değeri boş kabul et. Satırı silme,
   yalnızca o hücreyi boşalt.
4. Hedef sütunu boş olan satırları at; onlar için doğru cevabı bilmiyoruz.

Ekrana kaç sütun atıldığını, kaç yinelenen satır silindiğini ve kaç satır kaldığını
yazsın.

Sonra bu işlem parçasını kohort üzerinde çalıştır ve sonucu temiz adıyla sakla.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
veriyi_temizle adında çalıştırılabilir bir işlem parçası olmalı ve bir tablo döndürmeli.
temiz adında bir tablo hazır olmalı; hasta_id ve hedef sütunlarını içermeli.
temiz tablosunda hedef sütununda boş değer kalmamalı.
```


In [ ]:
#@cdss temizleme
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_function('veriyi_temizle', call_with=((kohort,), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(temiz, name='temiz', required=['hasta_id', 'hedef'], min_rows=30)

print('\nHedef sütununda boş değer:', int(temiz['hedef'].isna().sum()))


### Python notu · Eksik değer ve koşul

Gelen kodda `NaN` ifadesini göreceksiniz. Bu, *bilinmiyor* demektir; sıfır değildir.
Sıfır bir ölçümdür, `NaN` ölçümün yapılmamış olmasıdır. Klinik veride bu ayrım
belirleyicidir: Laktatı ölçülmemiş bir hasta ile laktatı sıfır çıkmış bir hasta aynı şey
değildir.

`if` ile başlayan satırlar **koşuldur**: Belirli bir durum sağlanıyorsa bir iş yapılır.
Üçüncü maddedeki yaş denetimi böyle yazılmıştır.

`df[df['yas'] < 120]` biçimindeki ifadeler **süzmedir**. İçteki kısım her satır için
doğru ya da yanlış üretir, dıştaki kısım yalnızca doğru olanları tutar. Tablodan satır
seçmenin en yaygın yolu budur.

Eksik değerlerin nasıl doldurulacağına henüz karar vermedik. O iş üçüncü adımda yapılacak
ve neden şimdi yapılmadığı orada anlaşılacak.


In [ ]:
# Temizlemeden sonra hangi sütunlarda ne kadar eksik kaldı.
eksik = temiz.isna().mean().sort_values(ascending=False)
print(eksik.head(8).round(3).to_string())


---

## Adım 2 · Eğitim ve sınama grubu

Model, elindeki veriden öğrenir. Aynı veriyle sınanırsa gerçekte ne kadar işe
yaradığını anlayamayız; ezberlediğini ölçmüş oluruz. Bu yüzden veri ikiye ayrılır.
Model bir grupla öğrenir, hiç görmediği diğer grupla sınanır.

Ayrımın nasıl yapıldığı kritiktir. NB1'in özetinde hasta sayısının satır sayısından
küçük olduğunu görmüştünüz: Bir hastanın birden fazla yatışı var. Satırlar rastgele
ayrılırsa aynı hastanın bir yatışı öğrenme grubuna, diğeri sınama grubuna düşer. Model o
hastayı tanır ve sınama sonucu olduğundan iyi çıkar.

**Ayrım hasta düzeyinde yapılmalıdır.** Bir hasta ya tamamen öğrenme grubunda ya tamamen
sınama grubunda olur. Yapay zekâ araçları bunu kendiliğinden yapmaz; açıkça istemeniz
gerekir.


### İstem 2

```
temiz tablosunu iki gruba ayır: Modelin öğreneceği grup ve modelin sınanacağı grup.

Ayrımı hasta düzeyinde yap. Aynı hastanın kayıtlarının tamamı tek bir grupta kalmalı;
bir hasta iki grupta birden bulunmamalı. Hasta kimliği hasta_id sütununda.

Sınama grubu verinin yaklaşık yüzde 30'u olsun. Tekrar üretilebilirlik için
RASTGELE_TOHUM değerini kullan.

Ekrana her iki grubun satır sayısını, hasta sayısını ve hedef durumun oranını yaz.
İki gruptaki oranlar birbirinden çok farklıysa uyar.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
egitim ve sinama adında iki tablo hazır olmalı.
Hiçbir hasta_id iki tabloda birden bulunmamalı.
Her iki tabloda da hedef sütunu hem 0 hem 1 değerini almalı.
```


In [ ]:
#@cdss ayrim
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2


In [ ]:
kit.check_split(egitim, sinama, target='hedef', patient_id='hasta_id')


### Python notu · Neden ayırıyoruz

Bir öğrencinin sınav sorularını önceden görmesi, sınavın ölçtüğü şeyi bozar. Model için
de durum aynıdır.

Sınama grubu, modelin hiç görmediği hastalardan oluşmalıdır. Aynı hastanın iki kaydı iki
gruba dağıldığında model o hastanın özelliklerini tanır ve sınavda yüksek not alır. Bu
not hastanede tekrarlanmaz.

Kontrol hücresi bunu doğrudan sınadı: İki gruptaki hasta kimliklerinin kesişimine baktı.
Kesişim boş değilse ayrım satır düzeyinde yapılmış demektir ve kod düzeltilmelidir.

Aynı sorun görüntü ve sinyal verisinde de vardır. Aynı hastadan alınmış iki görüntü ya da
iki sinyal parçası da iki gruba dağıtılmamalıdır.


---

## Adım 3 · Modelin anlayacağı biçime çevirmek

Model sayılarla çalışır. Tablomuzda ise cinsiyet, yatış türü ve sigorta gibi sayı
olmayan bilgiler var. Bunların sayıya çevrilmesi gerekir. Ayrıca eksik değerler
doldurulmalı ve sayısal sütunlar birbirine yakın ölçeklere getirilmelidir.

**Bu işlemlerin nasıl yapılacağı yalnızca öğrenme grubundan öğrenilir.** Örnek: Eksik
yaşları doldurmak için ortanca yaş kullanılacaksa, o ortanca yalnızca öğrenme grubundan
hesaplanır ve sınama grubuna aynen uygulanır.

Bu kural bu defterin en kolay atlanan noktasıdır. Ortanca bütün veriden hesaplanırsa
sınama grubundaki hastaların bilgisi öğrenme sürecine sızar. Sonuç yine sızıntıdır;
hata mesajı vermez, sınama sonucunu bir miktar yükseltir ve kod okunduğunda mantıklı
görünür.

Üçüncü adımın ayrımdan sonra gelmesinin sebebi budur.


### İstem 3

```
egitim ve sinama tablolarını modelin kullanabileceği biçime çevir.

Şunları yap:
1. hedef sütununu ayır; öğrenme grubunun hedefi y_egitim, sınama grubununki y_sinama
   olsun.
2. hasta_id ve stay_id gibi kimlik sütunlarını modele verme; bunlar hasta hakkında
   bilgi değil, numaradır.
3. Sayı olmayan sütunları sayıya çevir.
4. Eksik değerleri doldur.
5. Sayısal sütunları birbirine yakın ölçeklere getir.

Çok önemli: Üçüncü, dördüncü ve beşinci maddelerdeki dönüşümlerin nasıl yapılacağı
YALNIZCA öğrenme grubundan öğrenilsin, sonra sınama grubuna aynen uygulansın. Sınama
grubundan hiçbir bilgi öğrenme sürecine karışmasın. Bunu nasıl sağladığını açıklama
satırında yaz.

Ekrana iki grubun kaç satır ve kaç sütuna dönüştüğünü yaz.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
X_egitim, X_sinama, y_egitim, y_sinama adında dört nesne hazır olmalı.
X_egitim ile X_sinama aynı sayıda sütuna sahip olmalı.
X_egitim satır sayısı y_egitim uzunluğuna, X_sinama satır sayısı y_sinama uzunluğuna
eşit olmalı.
İçlerinde hiç boş değer kalmamalı.
```


In [ ]:
#@cdss hazirlama
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 3


In [ ]:
kit.check_defined('X_egitim', 'X_sinama', 'y_egitim', 'y_sinama')


In [ ]:
import numpy as _np

print('X_egitim boyutu :', _np.shape(X_egitim))
print('X_sinama boyutu :', _np.shape(X_sinama))
print('Sütun sayısı eşit mi   :', _np.shape(X_egitim)[1] == _np.shape(X_sinama)[1])
print('Satır sayıları uyuyor mu:',
      _np.shape(X_egitim)[0] == len(y_egitim) and _np.shape(X_sinama)[0] == len(y_sinama))
print('Boş değer kaldı mı     :', bool(_np.isnan(_np.asarray(X_egitim, dtype=float)).any()))


### Python notu · Öğrenmek ile uygulamak

Gelen kodda `fit` ve `transform` adında iki işlem göreceksiniz. Aradaki fark bu adımın
tamamıdır.

`fit` **öğrenir**: Ortancayı hesaplar, hangi kategorilerin bulunduğunu belirler,
ölçekleme için ortalama ve yayılımı çıkarır. `transform` ise **uygular**: Öğrenilmiş bu
değerleri kullanarak tabloyu çevirir.

Doğru kullanım şudur: Öğrenme grubunda `fit` ve `transform` birlikte çalışır, sınama
grubunda yalnızca `transform` çalışır. Sınama grubunda `fit` çağrılırsa sızıntı olur.

Kodunuzda `Pipeline` sözcüğünü de görebilirsiniz. Bu, bütün dönüşüm adımlarını tek bir
zincire bağlar ve zincirin tamamının yalnızca öğrenme grubunda öğrenilmesini sağlar.
Hatayı yapısal olarak engellediği için tercih edilen yoldur.

Kodunuzda sınama grubu üzerinde `fit` çağrısı var mı diye bakınız. Varsa yapay zekâ
aracına geri veriniz ve düzelttiriniz.


---

## Defter sonu · Kodun toplanması

Aşağıdaki hücre önceki defterlerden taşıdığınız kodla bu defterde eklediklerinizi tek
bir blok hâlinde toplar. Çıkan bloğun tamamını kopyalayınız; NB3 defterinin ilk
hücresine yapıştıracaksınız.

Blok ayrıca `cdss_nb2.py` adıyla kaydedilir. Colab oturumu kapandığında bu dosya silinir, bu
nedenle bloğu kendi bilgisayarınızda bir metin dosyasına da kopyalayınız.


In [ ]:
kod = kit.export(save_as='cdss_nb2.py')


## Bu defterde ne yapıldı

Sisteme üç katman eklendi: Temizleme, hasta düzeyinde ayrım ve modelin anlayacağı biçime
çevirme.

İki sızıntı yolu kapatıldı. Birincisi hasta düzeyinde ayrımla, ikincisi dönüşümlerin
yalnızca öğrenme grubundan öğrenilmesiyle. İkisi de hata mesajı vermeyen, sonucu
iyileştiren ve bu yüzden fark edilmesi güç hatalardır.

NB3'te bu verinin üzerine model kurulacak, öğretilecek ve başarımı ölçülecektir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelir ve Türkiye'deki bir yoğun
bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
